In [3]:
import os, time, random, json
import numpy as np, pandas as pd, torch, torch.nn as nn
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torch.amp import autocast, GradScaler
import torchvision
from torchvision.models import (resnet50, ResNet50_Weights, resnet18, ResNet18_Weights,
                                mobilenet_v3_small, MobileNet_V3_Small_Weights)
from torchvision import transforms as T
from PIL import Image
from pathlib import Path
from sklearn.metrics import f1_score, classification_report, confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

CFG = dict(
    data_dir="../data",            # paketteki klasör; Colab'de yolu güncelle
    out_dir="calismalar/nb1",
    model_name="resnet50",      # CPU'da: "mobilenet_v3_small" veya "resnet18"
    img_size=256, batch_size=64,
    epochs=20, patience=4,
    lr=1e-4, weight_decay=1e-4, label_smoothing=0.1,
    balance="weighted_loss",    # "none" | "weighted_loss" | "balanced_sampler"
    use_aug=True,               # D2 deneyi için False yap
    seed=42, num_workers=2, amp=True,
)

CANON = ["Chinee apple","Lantana","Parkinsonia","Parthenium","Prickly acacia",
         "Rubber vine","Siam weed","Snake weed","Negative"]

def set_seed(s):
    random.seed(s); np.random.seed(s)
    torch.manual_seed(s); torch.cuda.manual_seed_all(s)
    torch.backends.cudnn.deterministic = True; torch.backends.cudnn.benchmark = False

device = "cuda" if torch.cuda.is_available() else "cpu"
print("cihaz:", device, "| torch:", torch.__version__)

cihaz: cuda | torch: 2.6.0+cu124


In [4]:
DATA = CFG["data_dir"]
for root, dirs, files in os.walk(DATA):
    if root.count(os.sep) > 2: dirs[:] = []; continue
    print(root, "->", len(files), "dosya | alt klasörler:", dirs[:8], "| örnek:", files[:3])

train_df = pd.read_csv(os.path.join(DATA, "train.csv"))
val_df   = pd.read_csv(os.path.join(DATA, "val.csv"))
test_df  = pd.read_csv(os.path.join(DATA, "test.csv"))
print(train_df.head(3)); print("kolonlar:", train_df.columns.tolist())

../data -> 5 dosya | alt klasörler: ['images'] | örnek: ['dataset_card.json', 'LICENSE_NOTICE.md', 'test.csv']
../data\images -> 17509 dosya | alt klasörler: [] | örnek: ['T03_00000.jpg', 'T03_00001.jpg', 'T03_00002.jpg']
          id                  path  label    group_id              source_id  \
0  T03_00000  images/T03_00000.jpg      0  20160928_0  20160928-140314-0.jpg   
1  T03_00001  images/T03_00001.jpg      0  20160928_0  20160928-140337-0.jpg   
2  T03_00002  images/T03_00002.jpg      0  20160928_0  20160928-140731-0.jpg   

                                              sha256  
0  27027cb6d4a68a298f70c1b650541461d6dc5345cdb3f6...  
1  f59b22345665c84dc3efade60b07fef6a3c2b8b559fe5f...  
2  3886f6d831cb27a90bdb45f9bde11bcdf7ef3aa5a490d9...  
kolonlar: ['id', 'path', 'label', 'group_id', 'source_id', 'sha256']


In [5]:
IMG_CANDS = ["image","image_path","filepath","path","file","filename","img"]
LBL_CANDS = ["label","class","species","target","y","category"]

def detect_cols(df):
    cols = df.columns.tolist()
    img_col = next((c for c in IMG_CANDS if c in cols), None)
    if img_col is None:  # dosya adı içeren kolonu ara
        img_col = next((c for c in cols if df[c].astype(str).str.contains(
            r"\.(jpe?g|png)$", case=False).any()), cols[0])
    lbl_col = next((c for c in LBL_CANDS if c in cols and c != img_col), None)
    if lbl_col is None:
        num = [c for c in cols if c != img_col and pd.api.types.is_numeric_dtype(df[c])]
        lbl_col = num[-1] if num else None
    return img_col, lbl_col

img_col, lbl_col = detect_cols(train_df)
print("img_col:", img_col, "| lbl_col:", lbl_col)

path_by_name = {}
for p in Path(DATA).rglob("*"):
    if p.suffix.lower() in {".jpg",".jpeg",".png"}: path_by_name[p.name] = str(p)
print("indekslenen görüntü:", len(path_by_name))

def resolve(rel):
    p = os.path.join(DATA, str(rel))
    if os.path.exists(p): return p
    return path_by_name.get(os.path.basename(str(rel)))

for df, ad in [(train_df,"train"),(val_df,"val"),(test_df,"test")]:
    missing = df[img_col].map(lambda x: resolve(x) is None).sum()
    print(f"{ad}: eksik görüntü = {missing}")

def build_label_map(series):
    try:
        ints = sorted(int(v) for v in pd.unique(series))
        return {v: v for v in ints}                      # zaten 0..8
    except (ValueError, TypeError):
        pass
    m = {}
    for v in pd.unique(series):
        s = str(v).strip().lower()
        hit = next((i for i,c in enumerate(CANON) if s == c.lower() or s.startswith(c.lower()[:8])), None)
        m[v] = hit
    if all(x is not None for x in m.values()):
        return m                                          # brief sırasıyla 0..8
    print("UYARI: otomatik eşleşme olmadı — alfabetik sıra kullanıldı, doğrula!")
    return {v: i for i, v in enumerate(sorted(map(str, pd.unique(series))))}

label2id = build_label_map(train_df[lbl_col])
print("sınıf eşlemesi:", label2id)
print(train_df[lbl_col].value_counts())   # sınıf dengesizliğini gözle

img_col: path | lbl_col: label
indekslenen görüntü: 17509
train: eksik görüntü = 0
val: eksik görüntü = 0
test: eksik görüntü = 0
sınıf eşlemesi: {0: 0, 1: 1, 2: 2, 3: 3, 4: 4, 5: 5, 6: 6, 7: 7, 8: 8}
label
8    6496
7     857
4     814
0     798
3     772
5     667
1     630
2     580
6     495
Name: count, dtype: int64


In [6]:
MEAN, STD = [0.485,0.456,0.406], [0.229,0.224,0.225]

def train_tf(cfg):
    if cfg["use_aug"]:
        return T.Compose([
            T.RandomResizedCrop(cfg["img_size"], scale=(0.6,1.0)),
            T.RandomHorizontalFlip(), T.ColorJitter(0.2,0.2,0.2,0.02),
            T.ToTensor(), T.Normalize(MEAN, STD)])
    return T.Compose([T.Resize((cfg["img_size"],)*2), T.ToTensor(), T.Normalize(MEAN, STD)])

def eval_tf(cfg):
    return T.Compose([T.Resize((cfg["img_size"],)*2), T.ToTensor(), T.Normalize(MEAN, STD)])

class WeedDS(Dataset):
    def __init__(self, df, tf):
        self.df = df.reset_index(drop=True); self.tf = tf
    def __len__(self): return len(self.df)
    def __getitem__(self, i):
        r = self.df.iloc[i]
        x = Image.open(resolve(r[img_col])).convert("RGB")
        x = self.tf(x)
        y = label2id[r[lbl_col]] if lbl_col else -1
        return x, y

In [7]:
def build_model(name, n=9):
    if name == "resnet50":
        m = resnet50(weights=ResNet50_Weights.IMAGENET1K_V2)
        m.fc = nn.Linear(m.fc.in_features, n)
    elif name == "resnet18":
        m = resnet18(weights=ResNet18_Weights.IMAGENET1K_V1)
        m.fc = nn.Linear(m.fc.in_features, n)
    elif name == "mobilenet_v3_small":
        m = mobilenet_v3_small(weights=MobileNet_V3_Small_Weights.IMAGENET1K_V1)
        m.classifier[3] = nn.Linear(m.classifier[3].in_features, n)
    return m

In [ ]:
@torch.no_grad()
def predict_split(model, loader):
    model.eval(); ys, ps = [], []
    for x, y in loader:
        x = x.to(device, non_blocking=True)
        with autocast("cuda", enabled=(device=="cuda" and CFG["amp"])):
            logits = model(x)
        ps.append(torch.softmax(logits.float(), 1).cpu().numpy()); ys.append(y.numpy())
    return np.concatenate(ys), np.vstack(ps)

def train_one(cfg, train_df, val_df):
    set_seed(cfg["seed"]); os.makedirs(cfg["out_dir"], exist_ok=True)
    y_tr = train_df[lbl_col].map(label2id).values
    K = len(label2id)
    counts = np.bincount(y_tr, minlength=K).clip(min=1)

    use_amp = cfg["amp"] and device == "cuda"
    ds_tr = WeedDS(train_df, train_tf(cfg)); ds_va = WeedDS(val_df, eval_tf(cfg))

    if cfg["balance"] == "balanced_sampler":
        w_samp = torch.tensor([1.0/counts[y] for y in y_tr], dtype=torch.double)
        sampler = WeightedRandomSampler(w_samp, num_samples=len(ds_tr), replacement=True)
        dl_tr = DataLoader(ds_tr, cfg["batch_size"], sampler=sampler, shuffle=False,
                           num_workers=cfg["num_workers"], pin_memory=(device=="cuda"), drop_last=True)
        w_loss = None                       # sampler + weight çift düzeltme olur
    else:
        dl_tr = DataLoader(ds_tr, cfg["batch_size"], shuffle=True,
                           num_workers=cfg["num_workers"], pin_memory=(device=="cuda"), drop_last=True)
        w_loss = (torch.tensor(counts.sum()/(K*counts), dtype=torch.float32).to(device)
                  if cfg["balance"] == "weighted_loss" else None)

    dl_va = DataLoader(ds_va, cfg["batch_size"], shuffle=False,
                       num_workers=cfg["num_workers"], pin_memory=(device=="cuda"))

    model = build_model(cfg["model_name"], K).to(device)
    crit = nn.CrossEntropyLoss(weight=w_loss, label_smoothing=cfg["label_smoothing"])
    opt = torch.optim.AdamW(model.parameters(), lr=cfg["lr"], weight_decay=cfg["weight_decay"])
    sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=cfg["epochs"])
    scaler = GradScaler("cuda", enabled=use_amp)

    best_f1, best_state, bad, hist = 0.0, None, 0, []
    for ep in range(1, cfg["epochs"]+1):
        model.train(); tot = n = 0
        for x, y in dl_tr:
            x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
            opt.zero_grad(set_to_none=True)
            with autocast("cuda", enabled=use_amp):
                loss = crit(model(x), y)
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
            tot += loss.item()*len(y); n += len(y)
        sch.step()
        yv, pv = predict_split(model, dl_va); yhat = pv.argmax(1)
        f1, acc = f1_score(yv, yhat, average="macro"), (yv == yhat).mean()
        hist.append(dict(epoch=ep, loss=tot/n, val_f1=f1, val_acc=acc))
        print(f"ep {ep:02d} | loss {tot/n:.4f} | val macro-F1 {f1:.4f} | acc {acc:.4f}")
        if f1 > best_f1:
            best_f1, bad = f1, 0
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        else:
            bad += 1
            if bad >= cfg["patience"]: print("→ erken durdurma"); break

    ckpt = os.path.join(cfg["out_dir"], f"{cfg['model_name']}_{cfg['balance']}_seed{cfg['seed']}.pt")
    torch.save({"state_dict": best_state, "label2id": label2id, "cfg": cfg}, ckpt)
    print("EN İYİ val macro-F1:", round(best_f1, 4), "| checkpoint:", ckpt)
    return best_f1, ckpt, hist

best_f1, ckpt_path, hist = train_one(CFG, train_df, val_df)

Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to C:\Users\burak/.cache\torch\hub\checkpoints\resnet50-11ad3fa6.pth
100%|██████████| 97.8M/97.8M [00:19<00:00, 5.13MB/s]


In [ ]:
ck = torch.load(ckpt_path, map_location=device)
model = build_model(ck["cfg"]["model_name"], 9); model.load_state_dict(ck["state_dict"]); model.to(device)
yv, pv = predict_split(model, DataLoader(WeedDS(val_df, eval_tf(ck["cfg"])),
                       ck["cfg"]["batch_size"], num_workers=CFG["num_workers"]))
yhat = pv.argmax(1)

print(classification_report(yv, yhat, target_names=CANON, digits=4))
disp = ConfusionMatrixDisplay(confusion_matrix(yv, yhat), display_labels=CANON)
fig, ax = plt.subplots(figsize=(9,9)); disp.plot(ax=ax, xticks_rotation=45, values_format="d"); plt.show()

# en güvenilsiz hatalar (yanlış ama yüksek olasılıkla yanılanlar en öğretici)
paths = val_df[img_col].map(resolve).values
conf = pv.max(1); wrong = np.where(yv != yhat)[0]
wrong = wrong[np.argsort(-conf[wrong])][:12]
fig, axes = plt.subplots(3, 4, figsize=(14, 11))
for ax, i in zip(axes.ravel(), wrong):
    ax.imshow(Image.open(paths[i])); ax.axis("off")
    ax.set_title(f"T: {CANON[yv[i]]}\nP: {CANON[yhat[i]]} ({conf[i]:.2f})", fontsize=9)
plt.tight_layout(); plt.show()

In [ ]:
def write_predictions(ckpt_path, df, out_csv):
    ck = torch.load(ckpt_path, map_location=device)
    cfg, K = ck["cfg"], 9
    model = build_model(cfg["model_name"], K); model.load_state_dict(ck["state_dict"]); model.to(device)
    dl = DataLoader(WeedDS(df, eval_tf(cfg)), cfg["batch_size"], num_workers=CFG["num_workers"])
    model.eval(); probs, lat = [], []
    with torch.no_grad():
        for x, _ in dl:
            x = x.to(device, non_blocking=True)
            if device == "cuda": torch.cuda.synchronize()
            t0 = time.perf_counter()
            with autocast("cuda", enabled=(device=="cuda" and cfg["amp"])):
                p = torch.softmax(model(x).float(), 1)
            if device == "cuda": torch.cuda.synchronize()
            lat += [time.perf_counter()-t0] * len(x)
            probs.append(p.cpu().numpy())
    probs = np.vstack(probs); lat_ms = np.mean(lat) * 1000 / max(1, cfg["batch_size"] // cfg["batch_size"])
    id_col = "id" if "id" in df.columns else img_col
    out = pd.DataFrame({"id": df[id_col].values, "prediction": probs.argmax(1)})
    for k in range(K): out[f"prob_{k}"] = probs[:, k]
    out["latency_ms"] = round(lat_ms, 3)
    out["bytes"] = os.path.getsize(ckpt_path)
    out.to_csv(out_csv, index=False)
    assert np.allclose(out[[f"prob_{k}" for k in range(K)]].sum(1), 1, atol=1e-4)
    assert set(out["id"]) == set(df[id_col]), "id uyuşmazlığı!"
    print("yazıldı:", out_csv, "| satır:", len(out))

write_predictions(ckpt_path, val_df, "calismalar/nb1/dogrulama.csv")

In [ ]:
results = []
# D1: dengeleme stratejisi (aug, model, lr sabit)
for bal in ["none", "weighted_loss", "balanced_sampler"]:
    cfg = {**CFG, "balance": bal, "seed": 42}
    f1s, path, _ = train_one(cfg, train_df, val_df)
    results.append(dict(deney=f"D1: balance={bal}", val_macro_f1=round(f1s, 4)))
# D2: augmentation açık/kapalı (D1'in galibi ile aynı ayarlar)
cfg = {**CFG, "use_aug": False, "balance": "weighted_loss", "seed": 42}
f1s, path, _ = train_one(cfg, train_df, val_df)
results.append(dict(deney="D2: aug=off", val_macro_f1=round(f1s, 4)))
pd.DataFrame(results)

In [ ]:
# ÖNEMLİ: test'i SADECE tüm seçimler bitince çalıştır. Test üzerinde ayar yapılmaz.
write_predictions(ckpt_path, test_df, "calismalar/nb1/predictions.csv")